In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Data processing
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

# Traditional ML models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# Time series models
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet

# Deep learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Nixtla client for TimeGPT
from nixtla import NixtlaClient

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
df = pd.read_csv('../outputs/data/sales_weather_merged_filled_consolidated.csv')
print(f"Data shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Locations: {df['Location'].unique()}")

Data shape: (286, 18)
Date range: 2019-04-01 to 2024-03-01
Locations: ['Bangalore' 'Chennai' 'Cochin' 'Secunderabad' 'Vijaywada' 'Sricity']


In [3]:
df.groupby('Location')['QTY'].sum()

Location
Bangalore       225400.0
Chennai         783895.0
Cochin          219571.0
Secunderabad    652811.0
Sricity         118356.0
Vijaywada       224564.0
Name: QTY, dtype: float64

In [10]:
df.head()

,Location,QTY,temp_min,temp_max,temp_mean,dwpt_min,dwpt_max,dwpt_mean,rhum_min,rhum_max,rhum_mean,prcp_min,prcp_max,prcp_mean,wspd_min,wspd_max,wspd_mean,date
0,Bangalore,3951.0,19.00,35.250000,27.041071,0.775000,23.275000,15.193029,12.25,100.00,54.193651,0.0,7.766667,0.056111,0.0,36.625,10.347719,2019-05-01
1,Chennai,24969.0,25.10,39.900000,30.982535,17.800000,28.825000,25.123715,32.50,95.75,72.660069,0.0,2.700000,0.019583,0.0,36.575,11.148819,2019-05-01
2,Cochin,3218.0,23.00,35.666667,28.519028,19.233333,27.966667,24.551019,45.00,100.00,80.724537,0.0,10.200000,0.313796,1.0,29.000,7.195787,2019-05-01
3,Secunderabad,17294.0,20.75,39.750000,29.972061,6.650000,25.000000,17.374348,12.00,98.50,50.913699,0.0,3.933333,0.023148,1.7,28.250,10.836726,2019-05-01
4,Bangalore,4709.0,19.60,34.400000,26.270377,10.240000,24.420000,19.395948,24.80,100.00,69.900511,0.0,8.166667,0.159319,0.0,38.960,13.101580,2019-06-01


In [11]:
df_filtered = df[(df['date'] >= '2020-10-01') & df['Location'].isin(['Chennai', 'Cochin', 'Bangalore', 'Secunderabad'])].reset_index(drop=True)

In [12]:
# Create combined dataset by aggregating locations
def create_combined_dataset(df):
    """
    Combine all locations by summing QTY and averaging other features
    """
    print("Creating combined dataset...")
    
    # Group by date and aggregate
    combined_df = df.groupby('date').agg({
        'QTY': 'sum',  # Sum QTY across all locations
        'temp_mean': 'mean',  # Average temperature
        'temp_max': 'mean',   # Average max temperature
        'temp_min': 'mean',   # Average min temperature
        'rhum_mean': 'mean',  # Average humidity
        'wspd_mean': 'mean',  # Average wind speed
        'prcp_mean': 'mean',  # Average precipitation
    }).reset_index()
    return combined_df

# Create the combined dataset
combined_df = create_combined_dataset(df_filtered)
combined_df.head()


Creating combined dataset...


,date,QTY,temp_mean,temp_max,temp_min,rhum_mean,wspd_mean,prcp_mean
0,2020-10-01,21010.0,26.246574,33.0850,21.42625,83.151548,11.501288,0.300712
1,2020-11-01,16952.0,25.998580,32.6375,20.21125,81.203572,8.595730,0.286117
2,2020-12-01,19866.0,24.962569,31.7400,18.61125,82.426068,8.547400,0.303113
3,2021-01-01,35235.0,24.142288,31.4175,16.83750,78.438166,8.741769,0.139533
4,2021-02-01,24029.0,24.382607,32.2400,16.65500,73.167765,8.776383,0.041532


In [8]:
# Import helper functions from previous notebooks
def create_time_features(df):
    """
    Create comprehensive time-based features for forecasting
    """
    df = df.copy()
    
    # Extract temporal indicators
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    df['day_of_year'] = df['date'].dt.dayofyear
    df['week_of_year'] = df['date'].dt.isocalendar().week
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    df['is_quarter_start'] = df['date'].dt.is_quarter_start.astype(int)
    df['is_quarter_end'] = df['date'].dt.is_quarter_end.astype(int)
    
    # Cyclical encoding for seasonality
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
    df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)
    
    # Weather interaction features
    if 'temp_mean' in df.columns and 'rhum_mean' in df.columns:
        df['temp_humidity_interaction'] = df['temp_mean'] * df['rhum_mean']
    if 'temp_mean' in df.columns and 'wspd_mean' in df.columns:
        df['temp_wind_interaction'] = df['temp_mean'] * df['wspd_mean']
    
    return df

def create_lag_and_rolling_features(df, target_col='QTY', lags=[1, 2, 3, 6, 12], windows=[3, 6, 12]):
    """
    Create lag and rolling features to prevent data leakage
    """
    df = df.copy()
    
    # Create lag features (only using past data)
    for lag in lags:
        df[f'{target_col}_lag_{lag}'] = df[target_col].shift(lag)
    
    # Create rolling statistics (only using past data)
    for window in windows:
        df[f'{target_col}_rolling_mean_{window}'] = df[target_col].rolling(window=window, min_periods=1).mean().shift(1)
        df[f'{target_col}_rolling_std_{window}'] = df[target_col].rolling(window=window, min_periods=1).std().shift(1)
        df[f'{target_col}_rolling_min_{window}'] = df[target_col].rolling(window=window, min_periods=1).min().shift(1)
        df[f'{target_col}_rolling_max_{window}'] = df[target_col].rolling(window=window, min_periods=1).max().shift(1)
    
    return df

def prepare_data_for_modeling(df, test_months=6):
    """
    Prepare data for modeling with train/test split
    """

    # Create time features
    df = create_time_features(df)
    
    # Create lag and rolling features
    df = create_lag_and_rolling_features(df)
    
    # Split data: last test_months for testing
    split_idx = len(df) - test_months
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()
    
    # Remove rows with NaN from lag/rolling features in training set
    train_df = train_df.dropna().reset_index(drop=True)
    
    # Check if we have any training data left
    if len(train_df) == 0:
        print("Warning: No valid training data after feature engineering. Skipping.")
        return None
    
    # Define feature columns (exclude target and non-feature columns)
    exclude_cols = ['QTY', 'date']
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    
    # Prepare features and target
    X_train = train_df[feature_cols].fillna(0)  # Fill any remaining NaN with 0
    y_train = train_df['QTY']
    X_test = test_df[feature_cols].fillna(0)
    y_test = test_df['QTY']
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return {
        'X_train': X_train_scaled,
        'X_test': X_test_scaled,
        'y_train': y_train.values,
        'y_test': y_test.values,
        'train_df': train_df,
        'test_df': test_df,
        'feature_cols': feature_cols,
        'scaler': scaler
    }

def calculate_metrics(y_true, y_pred):
    """
    Calculate evaluation metrics
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    
    return {
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2
    }

print("Helper functions loaded successfully!")


Helper functions loaded successfully!


In [9]:
# Define model training functions
def create_ml_models():
    """Create a collection of ML models"""
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge': Ridge(alpha=1.0),
        'Lasso': Lasso(alpha=0.1),
        'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
        'AdaBoost': AdaBoostRegressor(n_estimators=100, random_state=42),
        'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42),
        'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
        'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=42, verbosity=-1),
        'CatBoost': CatBoostRegressor(iterations=100, random_state=42, verbose=False),
        'SVR': SVR(kernel='rbf'),
        'KNN': KNeighborsRegressor(n_neighbors=5)
    }
    return models

def train_ml_models(X_train, y_train, X_test, y_test):
    """Train all ML models and return predictions"""
    models = create_ml_models()
    results = {}
    
    for name, model in models.items():
        try:
            # Train model
            model.fit(X_train, y_train)
            
            # Make predictions
            y_pred = model.predict(X_test)
            
            # Calculate metrics
            metrics = calculate_metrics(y_test, y_pred)
            results[name] = {
                'model': model,
                'predictions': y_pred,
                'metrics': metrics
            }
            
            print(f"✓ {name}: MAE={metrics['MAE']:.2f}, RMSE={metrics['RMSE']:.2f}, R²={metrics['R2']:.3f}")
            
        except Exception as e:
            print(f"✗ {name}: Failed - {str(e)}")
    
    return results

def train_arima_model(train_df, test_df):
    """Train ARIMA model"""
    try:
        # Fit ARIMA model
        model = ARIMA(train_df['QTY'], order=(1, 1, 1))
        fitted_model = model.fit()
        
        # Make predictions
        predictions = fitted_model.forecast(steps=len(test_df))
        
        # Calculate metrics
        metrics = calculate_metrics(test_df['QTY'], predictions)
        
        print(f"✓ ARIMA: MAE={metrics['MAE']:.2f}, RMSE={metrics['RMSE']:.2f}, R²={metrics['R2']:.3f}")
        
        return {
            'model': fitted_model,
            'predictions': predictions,
            'metrics': metrics
        }
        
    except Exception as e:
        print(f"✗ ARIMA: Failed - {str(e)}")
        return None

def train_sarima_model(train_df, test_df):
    """Train SARIMA model"""
    try:
        # Fit SARIMA model
        model = SARIMAX(train_df['QTY'], order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
        fitted_model = model.fit(disp=False)
        
        # Make predictions
        predictions = fitted_model.forecast(steps=len(test_df))
        
        # Calculate metrics
        metrics = calculate_metrics(test_df['QTY'], predictions)
        
        print(f"✓ SARIMA: MAE={metrics['MAE']:.2f}, RMSE={metrics['RMSE']:.2f}, R²={metrics['R2']:.3f}")
        
        return {
            'model': fitted_model,
            'predictions': predictions,
            'metrics': metrics
        }
        
    except Exception as e:
        print(f"✗ SARIMA: Failed - {str(e)}")
        return None

def train_prophet_model(train_df, test_df):
    """Train Prophet model"""
    try:
        # Prepare data for Prophet
        prophet_train = train_df[['date', 'QTY']].copy()
        prophet_train.columns = ['ds', 'y']
        
        # Fit Prophet model
        model = Prophet()
        model.fit(prophet_train)
        
        # Make predictions
        future = model.make_future_dataframe(periods=len(test_df))
        forecast = model.predict(future)
        predictions = forecast['yhat'].iloc[-len(test_df):].values
        
        # Calculate metrics
        metrics = calculate_metrics(test_df['QTY'], predictions)
        
        print(f"✓ Prophet: MAE={metrics['MAE']:.2f}, RMSE={metrics['RMSE']:.2f}, R²={metrics['R2']:.3f}")
        
        return {
            'model': model,
            'predictions': predictions,
            'metrics': metrics
        }
        
    except Exception as e:
        print(f"✗ Prophet: Failed - {str(e)}")
        return None

def train_ets_model(train_df, test_df):
    """Train ETS model"""
    try:
        # Fit ETS model
        if len(train_df) < 24:
            # Use simple exponential smoothing if insufficient data
            model = ExponentialSmoothing(train_df['QTY'], seasonal=None)
        else:
            # Use seasonal Holt-Winters
            model = ExponentialSmoothing(train_df['QTY'], seasonal='add', seasonal_periods=12)
        
        fitted_model = model.fit()
        
        # Make predictions
        predictions = fitted_model.forecast(steps=len(test_df))
        
        # Calculate metrics
        metrics = calculate_metrics(test_df['QTY'], predictions)
        
        print(f"✓ ETS: MAE={metrics['MAE']:.2f}, RMSE={metrics['RMSE']:.2f}, R²={metrics['R2']:.3f}")
        
        return {
            'model': fitted_model,
            'predictions': predictions,
            'metrics': metrics
        }
        
    except Exception as e:
        print(f"✗ ETS: Failed - {str(e)}")
        return None

def train_timegpt_model(train_df, test_df):
    """Train TimeGPT model"""
    try:
        # Initialize Nixtla client
        client = NixtlaClient(api_key="nixak-BPWuiu0QLaDocnyGH7oFOutH821mnpHI5jFwgujKGyPGiLCAqNkQGUQ0vp11ZSOXX9msKcsCZgVM8cRu"
)
        
        # Prepare data for TimeGPT
        timegpt_train = train_df[['date', 'QTY']].copy()
        timegpt_train.columns = ['ds', 'y']
        timegpt_train['ds'] = pd.to_datetime(timegpt_train['ds'])
        timegpt_train = timegpt_train.set_index('ds').resample('MS').first().reset_index()
        timegpt_train = timegpt_train.dropna()
        
        if len(timegpt_train) < 12:
            print("✗ TimeGPT: Insufficient data")
            return None
        
        # Make predictions
        predictions = client.forecast(
            df=timegpt_train,
            h=len(test_df),
            freq='MS'
        )
        
        # Calculate metrics
        metrics = calculate_metrics(test_df['QTY'], predictions['TimeGPT'])
        
        print(f"✓ TimeGPT: MAE={metrics['MAE']:.2f}, RMSE={metrics['RMSE']:.2f}, R²={metrics['R2']:.3f}")
        
        return {
            'model': client,
            'predictions': predictions['TimeGPT'].values,
            'metrics': metrics
        }
        
    except Exception as e:
        print(f"✗ TimeGPT: Failed - {str(e)}")
        return None

print("Model training functions defined successfully!")


Model training functions defined successfully!


In [29]:
from sklearn.preprocessing import LabelEncoder
print("\n" + "="*80)
print("TRAINING MODELS")
print("="*80)

# Initialize results storage
individual_results = []

working_df = df_filtered.copy()
working_df['date'] = pd.to_datetime(working_df['date'])
combined_df['date'] = pd.to_datetime(combined_df['date'])

le = LabelEncoder()
working_df['Location'] = le.fit_transform(working_df['Location'])
print(le.classes_)
# Prepare data for location
working_data = prepare_data_for_modeling(working_df, 12)
combined_data = prepare_data_for_modeling(combined_df, 12)

print(f"Training Working samples: {len(working_data['y_train'])}")
print(f"Test Working samples: {len(working_data['y_test'])}")


print(f"Training Combined samples: {len(combined_data['y_train'])}")
print(f"Test Combined samples: {len(combined_data['y_test'])}")


TRAINING MODELS
['Bangalore' 'Chennai' 'Cochin' 'Secunderabad']
Training Working samples: 144
Test Working samples: 12
Training Combined samples: 18
Test Combined samples: 12


In [30]:

# Train ML models
print("\n--- Training ML Models ---")
working_ml_results = train_ml_models(
    working_data['X_train'],
    working_data['y_train'],
    working_data['X_test'],
    working_data['y_test']
)


combined_ml_results = train_ml_models(
    combined_data['X_train'],
    combined_data['y_train'],
    combined_data['X_test'],
    combined_data['y_test']
)


# Store ML results
for name, result in working_ml_results.items():
    individual_results.append({
        'Data_Type': 'Working',
        'Model_Type': 'ML',
        'Model_Name': name,
        'Model': result['model'],
        'MAE': result['metrics']['MAE'],
        'RMSE': result['metrics']['RMSE'],
        'MAPE': result['metrics']['MAPE'],
        'R2': result['metrics']['R2']
    })

for name, result in combined_ml_results.items():
    individual_results.append({
        'Data_Type': 'Combined',
        'Model_Type': 'ML',
        'Model_Name': name,
        'Model': result['model'],
        'MAE': result['metrics']['MAE'],
        'RMSE': result['metrics']['RMSE'],
        'MAPE': result['metrics']['MAPE'],
        'R2': result['metrics']['R2']
    })
# Train Time Series models
print("\n--- Training Time Series Models ---")

# ARIMA
working_arima_result = train_arima_model(working_data['train_df'], working_data['test_df'])
if working_arima_result:
    individual_results.append({
        'Data_Type': 'Working',
        'Model_Type': 'Time Series',
        'Model_Name': 'ARIMA',
        'Model': working_arima_result['model'],
        'MAE': working_arima_result['metrics']['MAE'],
        'RMSE': working_arima_result['metrics']['RMSE'],
        'MAPE': working_arima_result['metrics']['MAPE'],
        'R2': working_arima_result['metrics']['R2']
    })
combined_arima_result = train_arima_model(combined_data['train_df'], combined_data['test_df'])
if combined_arima_result:
    individual_results.append({
        'Data_Type': 'Combined',
        'Model_Type': 'Time Series',
        'Model_Name': 'ARIMA',
        'Model': combined_arima_result['model'],
        'MAE': combined_arima_result['metrics']['MAE'],
        'RMSE': combined_arima_result['metrics']['RMSE'],
        'MAPE': combined_arima_result['metrics']['MAPE'],
        'R2': combined_arima_result['metrics']['R2']
    })


# SARIMA
working_sarima_result = train_sarima_model(working_data['train_df'], working_data['test_df'])
if working_sarima_result:
    individual_results.append({
        'Data_Type': 'Working',
        'Model_Type': 'Time Series',
        'Model_Name': 'SARIMA',
        'Model': working_sarima_result['model'],
        'MAE': working_sarima_result['metrics']['MAE'],
        'RMSE': working_sarima_result['metrics']['RMSE'],
        'MAPE': working_sarima_result['metrics']['MAPE'],
        'R2': working_sarima_result['metrics']['R2']
    })
    
combined_sarima_result = train_sarima_model(combined_data['train_df'], combined_data['test_df'])
if combined_sarima_result:
    individual_results.append({
        'Data_Type': 'Combined',
        'Model_Type': 'Time Series',
        'Model_Name': 'SARIMA',
        'Model': combined_sarima_result['model'],
        'MAE': combined_sarima_result['metrics']['MAE'],
        'RMSE': combined_sarima_result['metrics']['RMSE'],
        'MAPE': combined_sarima_result['metrics']['MAPE'],
        'R2': combined_sarima_result['metrics']['R2']
    })

# Prophet
working_prophet_result = train_prophet_model(working_data['train_df'], working_data['test_df'])
if working_prophet_result:
    individual_results.append({
        'Data_Type': 'Working',
        'Model_Type': 'Time Series',
        'Model_Name': 'Prophet',
        'Model': working_prophet_result['model'],
        'MAE': working_prophet_result['metrics']['MAE'],
        'RMSE': working_prophet_result['metrics']['RMSE'],
        'MAPE': working_prophet_result['metrics']['MAPE'],
        'R2': working_prophet_result['metrics']['R2']
    })

combined_prophet_result = train_prophet_model(combined_data['train_df'], combined_data['test_df'])
if combined_prophet_result:
    individual_results.append({
        'Data_Type': 'Combined',
        'Model_Type': 'Time Series',
        'Model_Name': 'Prophet',
        'Model': combined_prophet_result['model'],
        'MAE': combined_prophet_result['metrics']['MAE'],
        'RMSE': combined_prophet_result['metrics']['RMSE'],
        'MAPE': combined_prophet_result['metrics']['MAPE'],
        'R2': combined_prophet_result['metrics']['R2']
    })

# ETS
working_ets_result = train_ets_model(working_data['train_df'], working_data['test_df'])
if working_ets_result:
    individual_results.append({
        'Data_Type': 'Working',
        'Model_Type': 'Time Series',
        'Model_Name': 'ETS',
        'Model': working_ets_result['model'],
        'MAE': working_ets_result['metrics']['MAE'],
        'RMSE': working_ets_result['metrics']['RMSE'],
        'MAPE': working_ets_result['metrics']['MAPE'],
        'R2': working_ets_result['metrics']['R2']
    })
    
combined_ets_result = train_ets_model(combined_data['train_df'], combined_data['test_df'])
if combined_ets_result:
    individual_results.append({
        'Data_Type': 'Combined',
        'Model_Type': 'Time Series',
        'Model_Name': 'ETS',
        'Model': combined_ets_result['model'],
        'MAE': combined_ets_result['metrics']['MAE'],   
        'RMSE': combined_ets_result['metrics']['RMSE'],
        'MAPE': combined_ets_result['metrics']['MAPE'],
        'R2': combined_ets_result['metrics']['R2']
    })



# TimeGPT
working_timegpt_result = train_timegpt_model(working_data['train_df'], working_data['test_df'])
if working_timegpt_result:
    individual_results.append({
        'Data_Type': 'Working',
        'Model_Type': 'Time Series',
        'Model_Name': 'TimeGPT',
        'Model': working_timegpt_result['model'],
        'MAE': working_timegpt_result['metrics']['MAE'],
        'RMSE': working_timegpt_result['metrics']['RMSE'],
        'MAPE': working_timegpt_result['metrics']['MAPE'],
        'R2': working_timegpt_result['metrics']['R2']
    })

combined_timegpt_result = train_timegpt_model(combined_data['train_df'], combined_data['test_df'])
if combined_timegpt_result:
    individual_results.append({
        'Data_Type': 'Combined',
        'Model_Type': 'Time Series',
        'Model_Name': 'TimeGPT',
        'Model': combined_timegpt_result['model'],
        'MAE': combined_timegpt_result['metrics']['MAE'],
        'RMSE': combined_timegpt_result['metrics']['RMSE'],
        'MAPE': combined_timegpt_result['metrics']['MAPE'],
        'R2': combined_timegpt_result['metrics']['R2']
    })



print(f"\n{'='*80}")
print(f"Individual location analysis completed!")
print(f"Total individual results: {len(individual_results)}")
print(f"{'='*80}")



--- Training ML Models ---
✓ Linear Regression: MAE=5178.62, RMSE=7881.23, R²=0.362
✓ Ridge: MAE=4497.66, RMSE=6914.61, R²=0.509
✓ Lasso: MAE=4792.64, RMSE=7555.39, R²=0.414
✓ ElasticNet: MAE=4429.75, RMSE=6732.91, R²=0.535
✓ Random Forest: MAE=4734.12, RMSE=7482.21, R²=0.425
✓ Gradient Boosting: MAE=4994.51, RMSE=8150.64, R²=0.318
✓ AdaBoost: MAE=4400.00, RMSE=7408.63, R²=0.437
✓ Extra Trees: MAE=5199.23, RMSE=8036.23, R²=0.337
✓ XGBoost: MAE=5135.02, RMSE=8171.69, R²=0.315
✓ LightGBM: MAE=5180.51, RMSE=7815.92, R²=0.373
✓ CatBoost: MAE=5414.14, RMSE=8401.37, R²=0.276
✓ SVR: MAE=8820.43, RMSE=13116.85, R²=-0.766
✓ KNN: MAE=5919.65, RMSE=8933.33, R²=0.181
✓ Linear Regression: MAE=33428.33, RMSE=36934.88, R²=-2.609
✓ Ridge: MAE=26627.82, RMSE=28456.33, R²=-1.142
✓ Lasso: MAE=34454.57, RMSE=39940.44, R²=-3.221
✓ ElasticNet: MAE=27229.49, RMSE=29059.84, R²=-1.234
✓ Random Forest: MAE=10745.94, RMSE=15710.88, R²=0.347
✓ Gradient Boosting: MAE=14316.92, RMSE=17359.31, R²=0.203
✓ AdaBoost: 

INFO:prophet:Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
INFO:cmdstanpy:Chain [1] start processing


✓ SARIMA: MAE=6014.12, RMSE=8812.49, R²=0.203
✓ SARIMA: MAE=16093.25, RMSE=18448.49, R²=0.100


INFO:cmdstanpy:Chain [1] done processing
INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
INFO:prophet:n_changepoints greater than number of observations. Using 13.
INFO:cmdstanpy:Chain [1] start processing
INFO:cmdstanpy:Chain [1] done processing


✓ Prophet: MAE=7694.29, RMSE=10425.12, R²=-0.116
✓ Prophet: MAE=14863.40, RMSE=22638.51, R²=-0.356
✓ ETS: MAE=6411.61, RMSE=9308.97, R²=0.111
✓ ETS: MAE=14872.62, RMSE=22652.63, R²=-0.358


INFO:nixtla.nixtla_client:Validating inputs...
INFO:nixtla.nixtla_client:Validating inputs...
INFO:nixtla.nixtla_client:Preprocessing dataframes...
INFO:nixtla.nixtla_client:Querying model metadata...


✗ TimeGPT: Failed - Series contain missing or duplicate timestamps, or the timestamps do not match the provided frequency.
Please make sure that all series have a single observation from the first to the last timestamp and that the provided frequency matches the timestamps'.
You can refer to https://docs.nixtla.io/docs/tutorials-missing_values for an end to end example.


INFO:nixtla.nixtla_client:Restricting input...
INFO:nixtla.nixtla_client:Calling Forecast Endpoint...


✓ TimeGPT: MAE=12845.78, RMSE=18751.65, R²=0.070

Individual location analysis completed!
Total individual results: 35


In [31]:
# Create comprehensive results comparison
print("\n" + "="*80)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*80)

# Create final results DataFrame
final_results_df = pd.DataFrame(individual_results)

# Sort by R² score (descending)
final_results_df = final_results_df.sort_values('R2', ascending=False)

print("\nTop 10 Models by R² Score:")
print("-" * 50)
top_10 = final_results_df.head(10)[['Data_Type', 'Model_Type', 'Model_Name', 'MAE', 'RMSE', 'R2']]
print(top_10.to_string(index=False))

print(f"\nTotal models evaluated: {len(final_results_df)}")
print(f"Model types included: {final_results_df['Model_Type'].unique()}")

# Summary by model type
print("\n" + "="*60)
print("SUMMARY BY MODEL TYPE")
print("="*60)

model_type_summary = final_results_df.groupby('Model_Type').agg({
    'MAE': ['mean', 'min'],
    'RMSE': ['mean', 'min'],
    'R2': ['mean', 'max']
}).round(3)

print(model_type_summary)



COMPREHENSIVE MODEL COMPARISON

Top 10 Models by R² Score:
--------------------------------------------------
Data_Type Model_Type        Model_Name          MAE         RMSE       R2
  Working         ML        ElasticNet  4429.753668  6732.908057 0.534703
  Working         ML             Ridge  4497.664451  6914.611136 0.509250
  Working         ML          AdaBoost  4399.999095  7408.629511 0.436622
  Working         ML     Random Forest  4734.124167  7482.211873 0.425375
  Working         ML             Lasso  4792.638469  7555.386233 0.414081
 Combined         ML       Extra Trees 11014.000000 15084.966290 0.397945
  Working         ML          LightGBM  5180.514659  7815.919975 0.372975
  Working         ML Linear Regression  5178.622913  7881.226378 0.362453
 Combined         ML     Random Forest 10745.944167 15710.884990 0.346946
  Working         ML       Extra Trees  5199.231667  8036.230684 0.337129

Total models evaluated: 35
Model types included: ['ML' 'Time Series']

SUM

In [40]:
base_individual_results = []

base_working_df = df_filtered.copy()
base_combined_df = combined_df.copy()
base_working_df['date'] = pd.to_datetime(base_working_df['date'])
base_combined_df['date'] = pd.to_datetime(combined_df['date'])

le = LabelEncoder()
base_working_df['Location'] = le.fit_transform(base_working_df['Location'])
print(le.classes_)

base_working_df = working_df[['date', 'QTY', 'Location']]
base_combined_df = base_combined_df[['date', 'QTY']]

['Bangalore' 'Chennai' 'Cochin' 'Secunderabad']


In [41]:
from sklearn.preprocessing import LabelEncoder
print("\n" + "="*80)
print("TRAINING MODELS")
print("="*80)

le = LabelEncoder()
base_working_df['Location'] = le.fit_transform(base_working_df['Location'])
print(le.classes_)
# Prepare data for location
base_working_data = prepare_data_for_modeling(base_working_df, 12)
base_combined_data = prepare_data_for_modeling(base_combined_df, 12)

print(f"Training Working samples: {len(base_working_data['y_train'])}")
print(f"Test Working samples: {len(base_working_data['y_test'])}")


print(f"Training Combined samples: {len(base_combined_data['y_train'])}")
print(f"Test Combined samples: {len(base_combined_data['y_test'])}")


TRAINING MODELS
[0 1 2 3]
Training Working samples: 144
Test Working samples: 12
Training Combined samples: 18
Test Combined samples: 12


In [42]:

# Train ML models
print("\n--- Training ML Models ---")
base_working_ml_results = train_ml_models(
    base_working_data['X_train'],
    base_working_data['y_train'],
    base_working_data['X_test'],
    base_working_data['y_test']
)


base_combined_ml_results = train_ml_models(
    base_combined_data['X_train'],
    base_combined_data['y_train'],
    base_combined_data['X_test'],
    base_combined_data['y_test']
)


# Store ML results
for name, result in base_working_ml_results.items():
    base_individual_results.append({
        'Data_Type': 'Working',
        'Model_Type': 'ML',
        'Model_Name': name,
        'Model': result['model'],
        'MAE': result['metrics']['MAE'],
        'RMSE': result['metrics']['RMSE'],
        'MAPE': result['metrics']['MAPE'],
        'R2': result['metrics']['R2']
    })

for name, result in base_combined_ml_results.items():
    base_individual_results.append({
        'Data_Type': 'Combined',
        'Model_Type': 'ML',
        'Model_Name': name,
        'Model': result['model'],
        'MAE': result['metrics']['MAE'],
        'RMSE': result['metrics']['RMSE'],
        'MAPE': result['metrics']['MAPE'],
        'R2': result['metrics']['R2']
    })
# Train Time Series models
print("\n--- Training Time Series Models ---")

# ARIMA
base_working_arima_result = train_arima_model(base_working_data['train_df'], base_working_data['test_df'])
if base_working_arima_result:
    base_individual_results.append({
        'Data_Type': 'Working',
        'Model_Type': 'Time Series',
        'Model_Name': 'ARIMA',
        'Model': base_working_arima_result['model'],
        'MAE': base_working_arima_result['metrics']['MAE'],
        'RMSE': base_working_arima_result['metrics']['RMSE'],
        'MAPE': base_working_arima_result['metrics']['MAPE'],
        'R2': base_working_arima_result['metrics']['R2']
    })
base_combined_arima_result = train_arima_model(base_combined_data['train_df'], base_combined_data['test_df'])
if base_combined_arima_result:
    base_individual_results.append({
        'Data_Type': 'Combined',
        'Model_Type': 'Time Series',
        'Model_Name': 'ARIMA',
        'Model': base_combined_arima_result['model'],
        'MAE': base_combined_arima_result['metrics']['MAE'],
        'RMSE': base_combined_arima_result['metrics']['RMSE'],
        'MAPE': base_combined_arima_result['metrics']['MAPE'],
        'R2': base_combined_arima_result['metrics']['R2']
    })


# SARIMA
base_working_sarima_result = train_sarima_model(base_working_data['train_df'], base_working_data['test_df'])
if base_working_sarima_result:
    base_individual_results.append({
        'Data_Type': 'Working',
        'Model_Type': 'Time Series',
        'Model_Name': 'SARIMA',
        'Model': base_working_sarima_result['model'],
        'MAE': base_working_sarima_result['metrics']['MAE'],
        'RMSE': base_working_sarima_result['metrics']['RMSE'],
        'MAPE': base_working_sarima_result['metrics']['MAPE'],
        'R2': base_working_sarima_result['metrics']['R2']
    })
    
base_combined_sarima_result = train_sarima_model(base_combined_data['train_df'], base_combined_data['test_df'])
if base_combined_sarima_result:
    base_individual_results.append({
        'Data_Type': 'Combined',
        'Model_Type': 'Time Series',
        'Model_Name': 'SARIMA',
        'Model': base_combined_sarima_result['model'],
        'MAE': base_combined_sarima_result['metrics']['MAE'],
        'RMSE': base_combined_sarima_result['metrics']['RMSE'],
        'MAPE': base_combined_sarima_result['metrics']['MAPE'],
        'R2': base_combined_sarima_result['metrics']['R2']
    })

# Prophet
base_working_prophet_result = train_prophet_model(base_working_data['train_df'], base_working_data['test_df'])
if base_working_prophet_result:
    base_individual_results.append({
        'Data_Type': 'Working',
        'Model_Type': 'Time Series',
        'Model_Name': 'Prophet',
        'Model': base_working_prophet_result['model'],
        'MAE': base_working_prophet_result['metrics']['MAE'],
        'RMSE': base_working_prophet_result['metrics']['RMSE'],
        'MAPE': base_working_prophet_result['metrics']['MAPE'],
        'R2': base_working_prophet_result['metrics']['R2']
    })

base_combined_prophet_result = train_prophet_model(base_combined_data['train_df'], base_combined_data['test_df'])
if base_combined_prophet_result:
    base_individual_results.append({
        'Data_Type': 'Combined',
        'Model_Type': 'Time Series',
        'Model_Name': 'Prophet',
        'Model': base_combined_prophet_result['model'],
        'MAE': base_combined_prophet_result['metrics']['MAE'],
        'RMSE': base_combined_prophet_result['metrics']['RMSE'],
        'MAPE': base_combined_prophet_result['metrics']['MAPE'],
        'R2': base_combined_prophet_result['metrics']['R2']
    })

# ETS
base_working_ets_result = train_ets_model(base_working_data['train_df'], base_working_data['test_df'])
if base_working_ets_result:
    base_individual_results.append({
        'Data_Type': 'Working',
        'Model_Type': 'Time Series',
        'Model_Name': 'ETS',
        'Model': base_working_ets_result['model'],
        'MAE': base_working_ets_result['metrics']['MAE'],
        'RMSE': base_working_ets_result['metrics']['RMSE'],
        'MAPE': base_working_ets_result['metrics']['MAPE'],
        'R2': base_working_ets_result['metrics']['R2']
    })
    
base_combined_ets_result = train_ets_model(base_combined_data['train_df'], base_combined_data['test_df'])
if base_combined_ets_result:
    base_individual_results.append({
        'Data_Type': 'Combined',
        'Model_Type': 'Time Series',
        'Model_Name': 'ETS',
        'Model': base_combined_ets_result['model'],
        'MAE': base_combined_ets_result['metrics']['MAE'],   
        'RMSE': base_combined_ets_result['metrics']['RMSE'],
        'MAPE': base_combined_ets_result['metrics']['MAPE'],
        'R2': base_combined_ets_result['metrics']['R2']
    })



# TimeGPT
base_working_timegpt_result = train_timegpt_model(base_working_data['train_df'], base_working_data['test_df'])
if base_working_timegpt_result:
    base_individual_results.append({
        'Data_Type': 'Working',
        'Model_Type': 'Time Series',
        'Model_Name': 'TimeGPT',
        'Model': base_working_timegpt_result['model'],
        'MAE': base_working_timegpt_result['metrics']['MAE'],
        'RMSE': base_working_timegpt_result['metrics']['RMSE'],
        'MAPE': base_working_timegpt_result['metrics']['MAPE'],
        'R2': base_working_timegpt_result['metrics']['R2']
    })

base_combined_timegpt_result = train_timegpt_model(base_combined_data['train_df'], base_combined_data['test_df'])
if base_combined_timegpt_result:
    base_individual_results.append({
        'Data_Type': 'Combined',
        'Model_Type': 'Time Series',
        'Model_Name': 'TimeGPT',
        'Model': base_combined_timegpt_result['model'],
        'MAE': base_combined_timegpt_result['metrics']['MAE'],
        'RMSE': base_combined_timegpt_result['metrics']['RMSE'],
        'MAPE': base_combined_timegpt_result['metrics']['MAPE'],
        'R2': base_combined_timegpt_result['metrics']['R2']
    })



print(f"\n{'='*80}")
print(f"Individual location analysis completed!")
print(f"Total individual results: {len(base_individual_results)}")
print(f"{'='*80}")



--- Training ML Models ---
✓ Linear Regression: MAE=3996.45, RMSE=6084.78, R²=0.620
✓ Ridge: MAE=4168.03, RMSE=6345.86, R²=0.587
✓ Lasso: MAE=4136.72, RMSE=6239.71, R²=0.600
✓ ElasticNet: MAE=4098.96, RMSE=6311.71, R²=0.591
✓ Random Forest: MAE=5804.14, RMSE=8149.01, R²=0.318
✓ Gradient Boosting: MAE=6333.96, RMSE=9235.27, R²=0.125
✓ AdaBoost: MAE=5283.15, RMSE=7881.09, R²=0.362
✓ Extra Trees: MAE=5320.37, RMSE=8274.43, R²=0.297
✓ XGBoost: MAE=6175.13, RMSE=9633.30, R²=0.047
✓ LightGBM: MAE=4940.90, RMSE=8189.03, R²=0.312
✓ CatBoost: MAE=5108.54, RMSE=8697.95, R²=0.223
✓ SVR: MAE=8818.85, RMSE=13116.02, R²=-0.766
✓ KNN: MAE=6113.67, RMSE=8636.85, R²=0.234
✓ Linear Regression: MAE=25605.77, RMSE=33349.32, R²=-1.943
✓ Ridge: MAE=21052.75, RMSE=25214.72, R²=-0.682
✓ Lasso: MAE=37637.19, RMSE=45332.62, R²=-4.437
✓ ElasticNet: MAE=21471.93, RMSE=25699.75, R²=-0.747
✓ Random Forest: MAE=10753.81, RMSE=15254.95, R²=0.384
✓ Gradient Boosting: MAE=14029.14, RMSE=16677.82, R²=0.264
✓ AdaBoost: 

INFO:prophet:Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
INFO:cmdstanpy:Chain [1] start processing


✓ SARIMA: MAE=6014.12, RMSE=8812.49, R²=0.203
✓ SARIMA: MAE=16093.25, RMSE=18448.49, R²=0.100


INFO:cmdstanpy:Chain [1] done processing
INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
INFO:prophet:n_changepoints greater than number of observations. Using 13.
INFO:cmdstanpy:Chain [1] start processing
INFO:cmdstanpy:Chain [1] done processing


✓ Prophet: MAE=7694.29, RMSE=10425.12, R²=-0.116
✓ Prophet: MAE=14863.40, RMSE=22638.51, R²=-0.356
✓ ETS: MAE=6411.61, RMSE=9308.97, R²=0.111
✓ ETS: MAE=14872.62, RMSE=22652.63, R²=-0.358


INFO:nixtla.nixtla_client:Validating inputs...
INFO:nixtla.nixtla_client:Validating inputs...
INFO:nixtla.nixtla_client:Preprocessing dataframes...
INFO:nixtla.nixtla_client:Querying model metadata...


✗ TimeGPT: Failed - Series contain missing or duplicate timestamps, or the timestamps do not match the provided frequency.
Please make sure that all series have a single observation from the first to the last timestamp and that the provided frequency matches the timestamps'.
You can refer to https://docs.nixtla.io/docs/tutorials-missing_values for an end to end example.


INFO:nixtla.nixtla_client:Restricting input...
INFO:nixtla.nixtla_client:Calling Forecast Endpoint...


✓ TimeGPT: MAE=12845.78, RMSE=18751.65, R²=0.070

Individual location analysis completed!
Total individual results: 35


In [43]:
# Create comprehensive results comparison
print("\n" + "="*80)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*80)

# Create final results DataFrame
final_results_df = pd.DataFrame(base_individual_results)

# Sort by R² score (descending)
final_results_df = final_results_df.sort_values('R2', ascending=False)

print("\nTop 10 Models by R² Score:")
print("-" * 50)
top_10 = final_results_df.head(10)[['Data_Type', 'Model_Type', 'Model_Name', 'MAE', 'RMSE', 'R2']]
print(top_10.to_string(index=False))

print(f"\nTotal models evaluated: {len(final_results_df)}")
print(f"Model types included: {final_results_df['Model_Type'].unique()}")

# Summary by model type
print("\n" + "="*60)
print("SUMMARY BY MODEL TYPE")
print("="*60)

model_type_summary = final_results_df.groupby('Model_Type').agg({
    'MAE': ['mean', 'min'],
    'RMSE': ['mean', 'min'],
    'R2': ['mean', 'max']
}).round(3)

print(model_type_summary)



COMPREHENSIVE MODEL COMPARISON

Top 10 Models by R² Score:
--------------------------------------------------
Data_Type Model_Type        Model_Name          MAE         RMSE       R2
  Working         ML Linear Regression  3996.448587  6084.783619 0.619973
  Working         ML             Lasso  4136.723901  6239.712658 0.600374
  Working         ML        ElasticNet  4098.962197  6311.713846 0.591098
  Working         ML             Ridge  4168.034105  6345.862967 0.586662
 Combined         ML       Extra Trees 10908.490000 14479.306477 0.445319
 Combined         ML     Random Forest 10753.812500 15254.948193 0.384300
 Combined         ML          AdaBoost 12231.736111 15482.737729 0.365775
  Working         ML          AdaBoost  5283.146147  7881.086883 0.362476
  Working         ML     Random Forest  5804.144167  8149.009036 0.318393
  Working         ML          LightGBM  4940.901718  8189.034496 0.311681

Total models evaluated: 35
Model types included: ['ML' 'Time Series']

SUM

In [57]:
test_df = df[['date', 'QTY', 'Location']]
test_df = df[['date', 'QTY', 'Location']]
test_df = test_df[test_df['Location'] == 'Sricity']
test_df['date'] = pd.to_datetime(test_df['date'])
test_df

,date,QTY,Location
243,2023-09-01,202.0,Sricity
249,2023-10-01,330.0,Sricity
255,2023-11-01,800.0,Sricity
261,2023-12-01,1600.0,Sricity
267,2024-01-01,34178.0,Sricity
273,2024-02-01,11676.0,Sricity
279,2024-03-01,14852.0,Sricity
285,2023-04-01,54718.0,Sricity


In [58]:
# Prepare test_df for prediction
print("Preparing test_df for prediction...")
print(f"Test data shape: {test_df.shape}")
print(f"Test data date range: {test_df['date'].min()} to {test_df['date'].max()}")

# Create time features for test_df
test_df_with_features = create_time_features(test_df)

# Create lag and rolling features for test_df
test_df_with_features = create_lag_and_rolling_features(test_df_with_features)

print(f"Test data with features shape: {test_df_with_features.shape}")
print(f"Feature columns: {[col for col in test_df_with_features.columns if col not in ['QTY', 'date']]}")

# Prepare features for prediction (exclude target and date)
exclude_cols = ['QTY', 'date']
feature_cols = [col for col in test_df_with_features.columns if col not in exclude_cols]
X_test_final = test_df_with_features[feature_cols].fillna(0)

print(f"Final test features shape: {X_test_final.shape}")
print(f"Test features columns: {feature_cols}")


Preparing test_df for prediction...
Test data shape: (8, 3)
Test data date range: 2023-04-01 00:00:00 to 2024-03-01 00:00:00
Test data with features shape: (8, 33)
Feature columns: ['Location', 'year', 'month', 'quarter', 'day_of_year', 'week_of_year', 'is_month_start', 'is_month_end', 'is_quarter_start', 'is_quarter_end', 'month_sin', 'month_cos', 'quarter_sin', 'quarter_cos', 'QTY_lag_1', 'QTY_lag_2', 'QTY_lag_3', 'QTY_lag_6', 'QTY_lag_12', 'QTY_rolling_mean_3', 'QTY_rolling_std_3', 'QTY_rolling_min_3', 'QTY_rolling_max_3', 'QTY_rolling_mean_6', 'QTY_rolling_std_6', 'QTY_rolling_min_6', 'QTY_rolling_max_6', 'QTY_rolling_mean_12', 'QTY_rolling_std_12', 'QTY_rolling_min_12', 'QTY_rolling_max_12']
Final test features shape: (8, 31)
Test features columns: ['Location', 'year', 'month', 'quarter', 'day_of_year', 'week_of_year', 'is_month_start', 'is_month_end', 'is_quarter_start', 'is_quarter_end', 'month_sin', 'month_cos', 'quarter_sin', 'quarter_cos', 'QTY_lag_1', 'QTY_lag_2', 'QTY_lag_3

In [59]:
# Run top models on test_df
print("\n" + "="*80)
print("RUNNING TOP MODELS ON TEST_DF")
print("="*80)

# Get the top 4 models from base_individual_results (Working data, ML models)
top_models = []
for result in base_individual_results:
    if result['Data_Type'] == 'Working' and result['Model_Type'] == 'ML':
        if result['Model_Name'] in ['Linear Regression', 'Lasso', 'ElasticNet', 'Ridge']:
            top_models.append(result)

# Sort by R² score
top_models.sort(key=lambda x: x['R2'], reverse=True)

print(f"Found {len(top_models)} top models to run:")
for i, model in enumerate(top_models, 1):
    print(f"{i}. {model['Model_Name']} - R² = {model['R2']:.3f}")

# Scale the test features using the same scaler from base_working_data
X_test_scaled = base_working_data['scaler'].transform(X_test_final)

print(f"\nScaled test features shape: {X_test_scaled.shape}")

# Generate predictions for each top model
predictions_results = []

for model_info in top_models:
    model_name = model_info['Model_Name']
    model = model_info['Model']
    
    try:
        # Generate predictions
        y_pred = model.predict(X_test_scaled)
        
        # Calculate metrics if we have actual values
        if len(test_df_with_features) > 0 and 'QTY' in test_df_with_features.columns:
            y_actual = test_df_with_features['QTY'].values
            metrics = calculate_metrics(y_actual, y_pred)
            
            predictions_results.append({
                'Model': model_name,
                'Predictions': y_pred,
                'Actual': y_actual,
                'Metrics': metrics
            })
            
            print(f"\n✓ {model_name} Predictions:")
            print(f"  MAE: {metrics['MAE']:.2f}")
            print(f"  RMSE: {metrics['RMSE']:.2f}")
            print(f"  MAPE: {metrics['MAPE']:.2f}%")
            print(f"  R²: {metrics['R2']:.3f}")
        else:
            predictions_results.append({
                'Model': model_name,
                'Predictions': y_pred,
                'Actual': None,
                'Metrics': None
            })
            
            print(f"\n✓ {model_name} Predictions Generated:")
            print(f"  Predictions: {y_pred}")
            
    except Exception as e:
        print(f"\n✗ {model_name}: Failed - {str(e)}")

print(f"\n{'='*80}")
print(f"Completed running {len(predictions_results)} top models on test_df")
print(f"{'='*80}")



RUNNING TOP MODELS ON TEST_DF
Found 4 top models to run:
1. Linear Regression - R² = 0.620
2. Lasso - R² = 0.600
3. ElasticNet - R² = 0.591
4. Ridge - R² = 0.587


ValueError: could not convert string to float: 'Sricity'

In [53]:
# Create comprehensive results summary
print("\n" + "="*80)
print("COMPREHENSIVE PREDICTION RESULTS")
print("="*80)

# Create a summary DataFrame
if predictions_results:
    summary_data = []
    for result in predictions_results:
        if result['Metrics'] is not None:
            summary_data.append({
                'Model': result['Model'],
                'MAE': result['Metrics']['MAE'],
                'RMSE': result['Metrics']['RMSE'],
                'MAPE': result['Metrics']['MAPE'],
                'R2': result['Metrics']['R2']
            })
    
    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        summary_df = summary_df.sort_values('R2', ascending=False)
        
        print("\nModel Performance Summary:")
        print("-" * 60)
        print(summary_df.to_string(index=False))
        
        # Find the best model
        best_model = summary_df.iloc[0]
        print(f"\n🏆 Best Model: {best_model['Model']}")
        print(f"   R² Score: {best_model['R2']:.3f}")
        print(f"   MAE: {best_model['MAE']:.2f}")
        print(f"   RMSE: {best_model['RMSE']:.2f}")

# Display detailed predictions
print("\n" + "="*60)
print("DETAILED PREDICTIONS")
print("="*60)

for i, result in enumerate(predictions_results, 1):
    print(f"\n{i}. {result['Model']}:")
    print(f"   Predictions: {result['Predictions']}")
    
    if result['Actual'] is not None:
        print(f"   Actual Values: {result['Actual']}")
        
        # Calculate prediction vs actual comparison
        pred_vs_actual = []
        for j, (pred, actual) in enumerate(zip(result['Predictions'], result['Actual'])):
            diff = pred - actual
            pct_diff = (diff / actual) * 100 if actual != 0 else 0
            pred_vs_actual.append(f"Point {j+1}: Pred={pred:.1f}, Actual={actual:.1f}, Diff={diff:.1f} ({pct_diff:+.1f}%)")
        
        print("   Detailed Comparison:")
        for comparison in pred_vs_actual:
            print(f"     {comparison}")

print(f"\n{'='*80}")
print("ANALYSIS COMPLETE")
print(f"{'='*80}")



COMPREHENSIVE PREDICTION RESULTS


NameError: name 'predictions_results' is not defined

In [54]:
# Create visualizations for actual vs predicted
print("\n" + "="*80)
print("CREATING ACTUAL VS PREDICTED PLOTS")
print("="*80)

import matplotlib.pyplot as plt
import seaborn as sns

# Set up the plotting style
plt.style.use('default')
sns.set_palette("husl")

# Create subplots for each model
if predictions_results and any(result['Actual'] is not None for result in predictions_results):
    
    # Determine the number of models to plot
    models_with_actuals = [result for result in predictions_results if result['Actual'] is not None]
    n_models = len(models_with_actuals)
    
    if n_models > 0:
        # Create subplots
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle('Actual vs Predicted Values - Top Models', fontsize=16, fontweight='bold')
        
        # Flatten axes for easier indexing
        axes = axes.flatten()
        
        for i, result in enumerate(models_with_actuals):
            if i < 4:  # Limit to 4 plots
                ax = axes[i]
                
                actual = result['Actual']
                predicted = result['Predictions']
                model_name = result['Model']
                
                # Create scatter plot
                ax.scatter(actual, predicted, alpha=0.7, s=100, edgecolors='black', linewidth=0.5)
                
                # Add perfect prediction line (y=x)
                min_val = min(min(actual), min(predicted))
                max_val = max(max(actual), max(predicted))
                ax.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8, linewidth=2, label='Perfect Prediction')
                
                # Calculate R² for this model
                if result['Metrics']:
                    r2 = result['Metrics']['R2']
                    mae = result['Metrics']['MAE']
                    ax.set_title(f'{model_name}\nR² = {r2:.3f}, MAE = {mae:.1f}', fontweight='bold')
                
                # Set labels and formatting
                ax.set_xlabel('Actual Values', fontweight='bold')
                ax.set_ylabel('Predicted Values', fontweight='bold')
                ax.grid(True, alpha=0.3)
                ax.legend()
                
                # Add text box with additional metrics
                if result['Metrics']:
                    rmse = result['Metrics']['RMSE']
                    mape = result['Metrics']['MAPE']
                    textstr = f'RMSE: {rmse:.1f}\nMAPE: {mape:.1f}%'
                    props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
                    ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=9,
                           verticalalignment='top', bbox=props)
        
        # Hide unused subplots
        for i in range(n_models, 4):
            axes[i].set_visible(False)
        
        plt.tight_layout()
        plt.show()
        
        # Create a combined plot showing all models together
        plt.figure(figsize=(12, 8))
        
        colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown']
        
        for i, result in enumerate(models_with_actuals):
            actual = result['Actual']
            predicted = result['Predictions']
            model_name = result['Model']
            
            plt.scatter(actual, predicted, alpha=0.7, s=100, 
                       label=f'{model_name} (R²={result["Metrics"]["R2"]:.3f})',
                       color=colors[i % len(colors)], edgecolors='black', linewidth=0.5)
        
        # Add perfect prediction line
        all_values = []
        for result in models_with_actuals:
            all_values.extend(result['Actual'])
            all_values.extend(result['Predictions'])
        
        min_val = min(all_values)
        max_val = max(all_values)
        plt.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.8, linewidth=2, label='Perfect Prediction')
        
        plt.xlabel('Actual Values', fontweight='bold', fontsize=12)
        plt.ylabel('Predicted Values', fontweight='bold', fontsize=12)
        plt.title('All Models: Actual vs Predicted Comparison', fontweight='bold', fontsize=14)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # Create time series plot if we have date information
        if 'date' in test_df_with_features.columns:
            plt.figure(figsize=(15, 8))
            
            dates = test_df_with_features['date']
            
            for i, result in enumerate(models_with_actuals):
                model_name = result['Model']
                predicted = result['Predictions']
                
                plt.plot(dates, predicted, marker='o', linewidth=2, markersize=6,
                        label=f'{model_name} (Predicted)', alpha=0.8)
            
            # Plot actual values
            actual = models_with_actuals[0]['Actual']  # All should have same actual values
            plt.plot(dates, actual, marker='s', linewidth=3, markersize=8,
                    label='Actual Values', color='black', alpha=0.9)
            
            plt.xlabel('Date', fontweight='bold', fontsize=12)
            plt.ylabel('Quantity', fontweight='bold', fontsize=12)
            plt.title('Time Series: Actual vs Predicted Values', fontweight='bold', fontsize=14)
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
        
        # Create residual plots
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle('Residual Plots - Top Models', fontsize=16, fontweight='bold')
        axes = axes.flatten()
        
        for i, result in enumerate(models_with_actuals):
            if i < 4:
                ax = axes[i]
                
                actual = result['Actual']
                predicted = result['Predictions']
                residuals = actual - predicted
                
                ax.scatter(predicted, residuals, alpha=0.7, s=100, edgecolors='black', linewidth=0.5)
                ax.axhline(y=0, color='r', linestyle='--', alpha=0.8)
                ax.set_xlabel('Predicted Values', fontweight='bold')
                ax.set_ylabel('Residuals (Actual - Predicted)', fontweight='bold')
                ax.set_title(f'{result["Model"]} Residuals', fontweight='bold')
                ax.grid(True, alpha=0.3)
        
        # Hide unused subplots
        for i in range(n_models, 4):
            axes[i].set_visible(False)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n✅ Created {4 if n_models >= 4 else n_models} visualization plots:")
        print("   1. Individual model scatter plots (Actual vs Predicted)")
        print("   2. Combined scatter plot (All models)")
        print("   3. Time series plot (Actual vs Predicted over time)")
        print("   4. Residual plots (Model error analysis)")
        
    else:
        print("❌ No models with actual values found for plotting")
        
else:
    print("❌ No prediction results available for plotting")

print(f"\n{'='*80}")
print("PLOTTING COMPLETE")
print(f"{'='*80}")



CREATING ACTUAL VS PREDICTED PLOTS


NameError: name 'predictions_results' is not defined

In [45]:
base_individual_results[0]['Model']

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False
